# 02 · Baselines and evaluation

A model is only good *relative to something*. This notebook fixes the **evaluation protocol** used across the whole lab (`tsforecasting.evaluation`). It also introduces the **naive benchmarks** that every other model must beat.

## Notation
We observe $y_1,\dots,y_T$ and forecast $\hat y_{T+h\mid T}$ for $h = 1,\dots,H$ (the *horizon*). $m$ denotes the seasonal period.

In [ ]:
%load_ext autoreload
%autoreload 2
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
from tsforecasting.data.data_loader import load_airline, load_jena, temporal_split
from tsforecasting.features.build_features import TARGET

try:
    df = load_jena()
except FileNotFoundError:  # first run: download + clean + features
    from tsforecasting.data import make_dataset
    from tsforecasting.features import build_features
    make_dataset.main(); build_features.main()
    df = load_jena()
y = df[TARGET]
airline = load_airline()
df.shape, airline.shape

## 1. Baselines (`tsforecasting.models.baselines`)

| method | forecast |
|---|---|
| naive | $\hat y_{T+h} = y_T$ |
| seasonal naive | $\hat y_{T+h} = y_{T+h-m\lceil h/m\rceil}$ |
| mean | $\hat y_{T+h} = \bar y$ |
| drift | $\hat y_{T+h} = y_T + h\,\dfrac{y_T - y_1}{T-1}$ |

All models in this lab follow one contract, `forecast_fn(y_train, h, X_train, X_future) -> DataFrame` with a `mean` column and optional quantile columns `q10…q90`.

In [ ]:
from tsforecasting.models import baselines
from tsforecasting.visualization.visualize import plot_forecast

train, test = airline.iloc[:-24], airline.iloc[-24:]
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=True)
for ax, (name, fn) in zip(axes.flat, {
    "naive": baselines.naive, "seasonal naive (m=12)": baselines.seasonal_naive(12),
    "mean": baselines.mean, "drift": baselines.drift}.items()):
    plot_forecast(train, fn(train, 24).set_index(test.index), test, title=name, ax=ax)
fig.tight_layout()

## 2. Error metrics

Let $e_h = y_{T+h} - \hat y_{T+h}$.

- **MAE** $= \frac1H\sum_h |e_h|$ and **RMSE** $= \sqrt{\frac1H\sum_h e_h^2}$ are scale-dependent and can't be compared across series. RMSE penalizes large errors more.
- **sMAPE** $= \frac{200}{H}\sum_h \frac{|e_h|}{|y_{T+h}|+|\hat y_{T+h}|}$ is scale-free but unstable near zero, which matters for Jena temperatures around $0\,^\circ$C.
- **MASE** (Hyndman & Koehler, 2006) scales by the in-sample seasonal-naive error:

$$\mathrm{MASE} = \frac{\frac1H\sum_h |e_h|}{\frac{1}{T-m}\sum_{t=m+1}^{T}|y_t - y_{t-m}|}.$$

MASE $<1$ means "better than the in-sample seasonal naive". It is our **headline metric**.

### Probabilistic forecasts
A point forecast hides uncertainty. For a quantile forecast $\hat y^{(q)}$, the **pinball (quantile) loss** is

$$L_q\big(y, \hat y^{(q)}\big) = \max\big(q\,(y-\hat y^{(q)}),\ (q-1)\,(y-\hat y^{(q)})\big).$$

Its expectation is minimized by the true $q$-quantile. We average it over $q\in\{0.1,\dots,0.9\}$, which approximates the CRPS. We also report the **coverage** of the $[q_{10}, q_{90}]$ interval, which should be $\approx 80\%$.

In [ ]:
from tsforecasting.evaluation import pinball_loss

e = np.linspace(-3, 3, 200)
for q in [0.1, 0.5, 0.9]:
    plt.plot(e, [pinball_loss([v], [0], q) for v in e], label=f"q = {q}")
plt.xlabel(r"$y - \hat y^{(q)}$"); plt.ylabel("loss"); plt.title("Pinball loss is asymmetric"); plt.legend();

## 3. Rolling-origin backtesting

A single train/test split gives one noisy number. **Rolling-origin evaluation** (time-series cross-validation) repeats the forecast from many cutoffs $c_k$. Each time it trains on $y_{1:c_k}$ and tests on $y_{c_k+1:c_k+H}$. The training window never sees the future.

In [ ]:
from tsforecasting.visualization.visualize import plot_backtest_folds

plot_backtest_folds(n=len(airline), initial=96, horizon=12, step=6).set_title(
    "Expanding window: initial=96, H=12, step=6");

In [ ]:
from tsforecasting.evaluation import backtest, score
from tsforecasting.visualization.visualize import plot_backtest, plot_leaderboard

fns = {"naive": baselines.naive, "seasonal_naive": baselines.seasonal_naive(12),
       "mean": baselines.mean, "drift": baselines.drift}
res = {k: backtest(f, airline, initial=96, horizon=12, step=6, m=12) for k, f in fns.items()}
board = pd.DataFrame({k: score(r) for k, r in res.items()}).T
display(board.round(3))
plot_leaderboard(board).set_title("Airline baselines (MASE, dashed line = 1)")
plot_backtest(res["seasonal_naive"], airline, title="seasonal naive across folds");

## 4. The Jena benchmark task

The same protocol on Jena: **forecast the next 24 h of temperature**, from 20 origins spread over the test period (last 10%), with $m=24$. This exact setup is reused by `make train` and notebook 08.

In [ ]:
from tsforecasting.models.train_model import HORIZON, N_FOLDS

train_df, val_df, test_df = temporal_split(df)
initial = len(train_df) + len(val_df)
step = (len(df) - initial - HORIZON) // N_FOLDS
jena_res = {k: backtest(f, y, initial, HORIZON, step, m=24)
            for k, f in {"naive": baselines.naive, "seasonal_naive": baselines.seasonal_naive(24)}.items()}
display(pd.DataFrame({k: score(r) for k, r in jena_res.items()}).T.round(3))
plot_backtest(jena_res["seasonal_naive"], y, title="Jena: seasonal naive, 24 h ahead");

## 5. Error grows with the horizon

Averaging over folds per step $h$ shows how quickly information decays. A forecaster worth using should bend this curve down.

In [ ]:
for k, r in jena_res.items():
    r.assign(ae=(r.y - r["mean"]).abs()).groupby("step").ae.mean().plot(label=k, marker=".")
plt.xlabel("horizon step h [hours]"); plt.ylabel("MAE [°C]"); plt.legend();

## Takeaways
- Seasonal naive is a strong baseline for strongly seasonal data, with MASE $\approx 0.9$ on Jena.
- Every model from now on is a `forecast_fn`. Same backtest, same metrics, comparable numbers.